# Student Performance Index Regression

Educational analytics regression model predicting student academic performance index from study habits and prior scores.

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [ ]:
df = pd.read_csv('student_performance_index_regression_data.csv')
print('Shape:', df.shape)
display(df.head())

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df['performance_index'], kde=True)
plt.title('Target Distribution: performance_index')
plt.tight_layout()
plt.show()

In [ ]:
X = df.drop(columns=['performance_index'])
y = df['performance_index']
num_cols = ['hours_studied', 'previous_scores', 'sleep_hours', 'sample_question_papers_practiced', 'attendance_rate']
cat_cols = ['extracurricular_activities']
preprocessor = ColumnTransformer([('num', StandardScaler(), num_cols), ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
models = {'Ridge': Ridge(alpha=1.0), 'RandomForestRegressor': RandomForestRegressor(n_estimators=150, random_state=42), 'GradientBoostingRegressor': GradientBoostingRegressor(random_state=42)}
best_score, best_pipe, best_name, results = -1e9, None, '', {}
for name, reg in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', reg)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    results[name] = {'r2': round(r2_score(y_test, preds), 4), 'mae': round(mean_absolute_error(y_test, preds), 2), 'rmse': round(np.sqrt(mean_squared_error(y_test, preds)), 2)}
    if results[name]['r2'] > best_score:
        best_score, best_pipe, best_name = results[name]['r2'], pipe, name
print('Best Model:', best_name)
print(pd.DataFrame(results).T)

In [ ]:
joblib.dump(best_pipe, 'best_model.pkl')
print('Saved best_model.pkl successfully.')